### Structured output

Structured output allows agents to return data in a specific, predictable format. Instead of parsing natural language responses, you get structured data in the form of JSON objects, Pydantic models, or dataclasses that your application can use directly.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:

import os
from dotenv import load_dotenv

load_dotenv()

# os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['GEMINI_API_KEY']=os.getenv('GEMINI_API_KEY')

In [2]:
import os
from langchain.chat_models import init_chat_model
# model = init_chat_model("groq:qwen/qwen3.6-27b")
model = init_chat_model("groq:openai/gpt-oss-120b")
response=model.invoke("Hellow, how are you?")
print(response)

content="Hello! I'm doing great, thank you for asking. How can I assist you today?" additional_kwargs={'reasoning_content': "We need to respond. The system message says we are ChatGPT. There's no special instruction besides being helpful. So just respond politely."} response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 78, 'total_tokens': 133, 'completion_time': 0.114911243, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.003218448, 'prompt_tokens_details': None, 'queue_time': 0.393458321, 'total_time': 0.118129691}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a07a1e-9de6-7f92-a3a7-18648b4fa0ba-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 78, 'output_tokens': 55, 'total_tokens': 133, 'output_token_details': {'reasoning': 28}}


In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie.")
    year: int = Field(..., description="The release year of the movie.")
    director: str = Field(..., description="The director of the movie.")
    rating: float = Field(..., description="The rating of the movie.")


In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure


_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000252639D2A50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000252639D3770>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The t

In [5]:
model_with_structure.invoke("Provide details of the movie Titanic")

Movie(title='Titanic', year=1997, director='James Cameron', rating=7.8)

#### Message output alongside past structure

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie.")
    year: int = Field(..., description="The release year of the movie.")
    director: str = Field(..., description="The director of the movie.")
    rating: float = Field(..., description="The rating of the movie.")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details of the movie Titanic")
response



{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "Provide details of the movie Titanic". We need to output details: director, rating, title, year. Use the function Movie. So we should call functions.Movie with appropriate arguments.\n\nTitanic (1997) directed by James Cameron, rating maybe IMDb rating 7.8? Could use typical rating. Provide rating as number. Provide title "Titanic". Year 1997.\n\nThus call function.', 'tool_calls': [{'id': 'fc_fbd73437-4b79-49f8-85bd-088a3f5d4ea0', 'function': {'arguments': '{"director":"James Cameron","rating":7.8,"title":"Titanic","year":1997}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 138, 'prompt_tokens': 155, 'total_tokens': 293, 'completion_time': 0.2893144, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 0.042883229, 'prompt_tokens_details': None, 'queue_time': 0.403534491, 'total_time': 0.332197629}, 'model_name': 'openai/gpt-oss-120b',

#### Nested structure

In [7]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    generes: list[str]
    budget: float | None = Field(None, description="The budget of the movie in million USD.")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details of the movie Titanic")
response

MovieDetails(title='Titanic', year=1997, cast=[Actor(name='Leonardo DiCaprio', role='Jack Dawson'), Actor(name='Kate Winslet', role='Rose DeWitt Bukater'), Actor(name='Billy Zane', role='Cal Hockley'), Actor(name='Kathy Bates', role='Molly Brown')], generes=['Drama', 'Romance', 'Disaster'], budget=None)

#### Typed dict

Typed dict provides a simple alternative usinf python's builtin typing, ideal when you don't need runtime validation

In [9]:
from typing_extensions import Annotated, TypedDict

class MovieDict(TypedDict):
    """ A movie with a title, year, director, and rating."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The release year"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie"]

In [10]:
model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Provide details of the movie Titanic")
response

{'director': 'James Cameron', 'rating': 7.8, 'title': 'Titanic', 'year': 1997}

In [11]:
# from pydantic import BaseModel, Field

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    generes: list[str]
    budget: float | None = Field(None, description="The budget of the movie in million USD.")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details of the movie Titanic")
response

{'budget': 200000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Jack Dawson'},
  {'name': 'Kate Winslet', 'role': 'Rose DeWitt Bukater'},
  {'name': 'Billy Zane', 'role': 'Cal Hockley'},
  {'name': 'Kathy Bates', 'role': 'Molly Brown'},
  {'name': 'Frances Fisher', 'role': 'Ruth Dewitt Bukater'},
  {'name': 'Bill Paxton', 'role': 'Brock Lovett'},
  {'name': 'Gloria Stuart', 'role': 'Old Rose'}],
 'generes': ['Drama', 'Romance', 'Disaster'],
 'title': 'Titanic',
 'year': 1997}

In [13]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

#### Data classes

A data class is a class typically containing mainly data, lathoigh there aren't really any restrictions. You create it using the @dataclass decorator

In [16]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """ Contact information for a person."""
    name: str = Field(..., description="The name of the person.")
    email: str = Field(..., description="The email address of the person.")
    phone: str = Field(..., description="The phone number of the person.")

agent = create_agent(
    model = model,
    response_format=ContactInfo, #Auto selects ProviderStrategy 
)

result = agent.invoke({"messages" : [{"role": "user", "content": "Extract the contact information from : john Doe, jhon@example.com, (555) 555-5555"}]})
print(result["structured_response"])

name='john Doe' email='jhon@example.com' phone='(555) 555-5555'


In [ ]:
# from pydantic import BaseModel, Field
from typing_extensions import Annotated, TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """ Contact information for a person."""
    name: str = Field(..., description="The name of the person.")
    email: str = Field(..., description="The email address of the person.")
    phone: str = Field(..., description="The phone number of the person.")

agent = create_agent(
    model = model,
    response_format=ContactInfo, #Auto selects ProviderStrategy 
)

result = agent.invoke({"messages" : [{"role": "user", "content": "Extract the contact information from : john Doe, jhon@example.com, (555) 555-5555"}]})
print(result["structured_response"])